# Capítulo 6 · Atención y Transformer

Un GPT pequeño, de caracteres, entrenado con el *Quijote*. Usa PyTorch en CPU y no necesita GPU. Ejecuta las celdas en orden; el entrenamiento tarda unos seis minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio: el mismo programa, por partes.

## 1. La atención a mano

Comprueba el ejemplo del capítulo: tres posiciones con sus consultas, claves y valores. La máscara pone −∞ en el futuro y softmax convierte las puntuaciones en pesos.

In [1]:
import torch
consultas = torch.tensor([[0., 1.], [0., 1.], [1., 0.]])
claves = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
valores = torch.tensor([[2., 0.], [0., 2.], [1., 1.]])
puntuaciones = consultas @ claves.T / 2 ** 0.5
mascara = torch.tril(torch.ones(3, 3))
puntuaciones = puntuaciones.masked_fill(mascara == 0, float("-inf"))
pesos = torch.softmax(puntuaciones, dim=-1)
print("Pesos:\n", pesos.round(decimals=3))
print("Salidas:\n", (pesos @ valores).round(decimals=3))

Pesos:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.3300, 0.6700, 0.0000],
        [0.4010, 0.1980, 0.4010]])
Salidas:
 tensor([[2.0000, 0.0000],
        [0.6600, 1.3400],
        [1.2030, 0.7970]])


## 2. Los datos y los lotes

En Colab, la primera celda descarga la novela si no está.

In [2]:
"""Capítulo 6: un GPT pequeño entrenado con el Quijote. PyTorch en CPU."""
import math
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

TEXTO = Path("material/quijote.txt")
if not TEXTO.exists():  # en Colab, descargamos la novela del repositorio
    import urllib.request
    TEXTO.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quijote.txt", TEXTO)
CONTEXTO, DIMENSION, CABEZAS, CAPAS = 64, 64, 4, 4
LOTE, PASOS, TASA, SEMILLA = 32, 3000, 3e-3, 1337


def preparar(texto):
    simbolos = sorted(set(texto))
    a_id = {c: i for i, c in enumerate(simbolos)}
    datos = torch.tensor([a_id[c] for c in texto], dtype=torch.long)
    corte = int(0.9 * len(datos))
    return simbolos, a_id, datos[:corte], datos[corte:]


def lote(datos, generador):
    tope = len(datos) - CONTEXTO - 1
    inicios = torch.randint(tope, (LOTE,), generator=generador)
    x = torch.stack([datos[i:i + CONTEXTO] for i in inicios])
    y = torch.stack([datos[i + 1:i + CONTEXTO + 1] for i in inicios])
    return x, y

## 3. Una cabeza, un bloque y el modelo

In [3]:
class Cabeza(nn.Module):
    """Una cabeza de atención causal."""

    def __init__(self, tamano):
        super().__init__()
        self.consulta = nn.Linear(DIMENSION, tamano, bias=False)
        self.clave = nn.Linear(DIMENSION, tamano, bias=False)
        self.valor = nn.Linear(DIMENSION, tamano, bias=False)
        mascara = torch.tril(torch.ones(CONTEXTO, CONTEXTO))
        self.register_buffer("mascara", mascara)

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.consulta(x), self.clave(x), self.valor(x)
        puntuaciones = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        futuro = self.mascara[:T, :T] == 0
        puntuaciones = puntuaciones.masked_fill(futuro, float("-inf"))
        pesos = F.softmax(puntuaciones, dim=-1)
        self.ultimos_pesos = pesos.detach()  # para poder mirarlos después
        return pesos @ v

In [4]:
class Bloque(nn.Module):
    """Atención entre posiciones seguida de una MLP en cada posición."""

    def __init__(self):
        super().__init__()
        tamano = DIMENSION // CABEZAS
        self.cabezas = nn.ModuleList([Cabeza(tamano) for _ in range(CABEZAS)])
        self.proyeccion = nn.Linear(DIMENSION, DIMENSION)
        self.mlp = nn.Sequential(nn.Linear(DIMENSION, 4 * DIMENSION), nn.ReLU(),
                                 nn.Linear(4 * DIMENSION, DIMENSION))
        self.norma1 = nn.LayerNorm(DIMENSION)
        self.norma2 = nn.LayerNorm(DIMENSION)

    def forward(self, x):
        h = self.norma1(x)
        atencion = torch.cat([cabeza(h) for cabeza in self.cabezas], dim=-1)
        x = x + self.proyeccion(atencion)
        x = x + self.mlp(self.norma2(x))
        return x

In [5]:
class GPT(nn.Module):
    def __init__(self, vocabulario):
        super().__init__()
        self.embedding_simbolos = nn.Embedding(vocabulario, DIMENSION)
        self.embedding_posiciones = nn.Embedding(CONTEXTO, DIMENSION)
        self.bloques = nn.Sequential(*[Bloque() for _ in range(CAPAS)])
        self.norma = nn.LayerNorm(DIMENSION)
        self.salida = nn.Linear(DIMENSION, vocabulario)

    def forward(self, ids):
        T = ids.shape[1]
        posiciones = torch.arange(T)
        x = self.embedding_simbolos(ids) + self.embedding_posiciones(posiciones)
        return self.salida(self.norma(self.bloques(x)))

In [6]:
torch.manual_seed(SEMILLA)
simbolos, a_id, entrenamiento, validacion = preparar(TEXTO.read_text(encoding="utf-8"))
prueba = GPT(len(simbolos))
print(len(simbolos), "símbolos ·", sum(p.numel() for p in prueba.parameters()), "parámetros")
x, y = lote(entrenamiento, torch.Generator().manual_seed(0))
print("Lote:", tuple(x.shape), "→ logits:", tuple(prueba(x).shape))

92 símbolos · 215260 parámetros
Lote: (32, 64) → logits: (32, 64, 92)


## 4. Evaluar y entrenar

In [7]:
def perdida_media(modelo, x, y):
    logits = modelo(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def evaluar(modelo, datos, lotes=20):
    generador = torch.Generator().manual_seed(0)
    perdidas = []
    for _ in range(lotes):
        x, y = lote(datos, generador)
        perdidas.append(perdida_media(modelo, x, y).item())
    return sum(perdidas) / len(perdidas)


def entrenar(modelo, entrenamiento, validacion):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=TASA)
    generador = torch.Generator().manual_seed(SEMILLA)
    historial = []
    for paso in range(PASOS + 1):
        if paso % 250 == 0:
            medida = (paso, evaluar(modelo, entrenamiento),
                      evaluar(modelo, validacion))
            historial.append(medida)
            print("Paso %4d · entrenamiento %.3f · validación %.3f" % medida)
        if paso == PASOS:
            break
        x, y = lote(entrenamiento, generador)
        perdida = perdida_media(modelo, x, y)
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
    return historial

## 5. Generar, comprobar la máscara y ejecutar

La última celda entrena el modelo completo: tarda unos seis minutos.

In [8]:
@torch.no_grad()
def generar(modelo, simbolos, a_id, inicio, n, generador):
    ids = torch.tensor([[a_id[c] for c in inicio]], dtype=torch.long)
    for _ in range(n):
        logits = modelo(ids[:, -CONTEXTO:])[:, -1, :]
        probabilidades = F.softmax(logits, dim=-1)
        siguiente = torch.multinomial(probabilidades, 1, generator=generador)
        ids = torch.cat([ids, siguiente], dim=1)
    return "".join(simbolos[i] for i in ids[0].tolist())


@torch.no_grad()
def comprobar_mascara(modelo, datos, posicion=CONTEXTO // 2):
    x = datos[:CONTEXTO].unsqueeze(0)
    cambiado = x.clone()
    cambiado[0, posicion] = (x[0, posicion] + 1) % modelo.salida.out_features
    antes, despues = modelo(x), modelo(cambiado)
    pasado_igual = torch.allclose(antes[0, :posicion], despues[0, :posicion])
    resto_cambia = not torch.allclose(antes[0, posicion:],
                                      despues[0, posicion:])
    return pasado_igual, resto_cambia


def bigrama(entrenamiento, validacion, vocabulario):
    cuentas = torch.ones(vocabulario, vocabulario)
    unos = torch.ones(len(entrenamiento) - 1)
    parejas = (entrenamiento[:-1], entrenamiento[1:])
    cuentas.index_put_(parejas, unos, accumulate=True)
    tabla = cuentas / cuentas.sum(dim=1, keepdim=True)
    return -tabla[validacion[:-1], validacion[1:]].log().mean().item()


def ejecutar():
    torch.set_num_threads(4)
    torch.manual_seed(SEMILLA)
    texto = TEXTO.read_text(encoding="utf-8")
    simbolos, a_id, entrenamiento, validacion = preparar(texto)
    modelo = GPT(len(simbolos))
    n_parametros = sum(p.numel() for p in modelo.parameters())
    print("Símbolos:", len(simbolos), "· parámetros:", n_parametros)
    referencia = bigrama(entrenamiento, validacion, len(simbolos))
    print("Bigrama de caracteres, validación: %.3f" % referencia)
    sorteo = torch.Generator().manual_seed(7)
    sopa = generar(modelo, simbolos, a_id, "En un lugar", 60, sorteo)
    print("Muestra antes de entrenar:", repr(sopa))
    inicio = time.perf_counter()
    historial = entrenar(modelo, entrenamiento, validacion)
    print("Entrenamiento: %.0f segundos" % (time.perf_counter() - inicio))
    mascara = comprobar_mascara(modelo, validacion)
    print("Máscara causal (pasado intacto, resto cambia):", mascara)
    sorteo = torch.Generator().manual_seed(7)
    print(generar(modelo, simbolos, a_id, "En un lugar", 400, sorteo))
    return modelo, historial

In [9]:
modelo, historial = ejecutar()

Símbolos: 92 · parámetros: 215260
Bigrama de caracteres, validación: 2.305
Muestra antes de entrenar: 'En un lugaríU6ùní1aJG«àSù¿PG"rS¿cx«;\nJBéq6ZñrUR]ÉÍS\'-."¿¡—hùÁsÚc—X-ñÑ—F'
Paso    0 · entrenamiento 4.613 · validación 4.616
Paso  250 · entrenamiento 2.151 · validación 2.165
Paso  500 · entrenamiento 1.886 · validación 1.890
Paso  750 · entrenamiento 1.740 · validación 1.747
Paso 1000 · entrenamiento 1.658 · validación 1.666
Paso 1250 · entrenamiento 1.603 · validación 1.616
Paso 1500 · entrenamiento 1.562 · validación 1.587
Paso 1750 · entrenamiento 1.530 · validación 1.556
Paso 2000 · entrenamiento 1.499 · validación 1.528
Paso 2250 · entrenamiento 1.484 · validación 1.509
Paso 2500 · entrenamiento 1.469 · validación 1.495
Paso 2750 · entrenamiento 1.457 · validación 1.487
Paso 3000 · entrenamiento 1.447 · validación 1.477
Entrenamiento: 344 segundos
Máscara causal (pasado intacto, resto cambia): (True, True)
En un lugar se nuda Golarde.

— Seo; y él sería, será cuando serás —dij

## 6. Experimenta

Genera más texto con otro comienzo o cambia la semilla del sorteo. Para el ejercicio 12, cambia la posición de `comprobar_mascara`.

In [10]:
print(generar(modelo, simbolos, a_id, "Sancho Panza", 300, torch.Generator().manual_seed(3)))
print(comprobar_mascara(modelo, validacion, posicion=10))

Sancho Panza trailación dellos caballerinos y
vedo los mentidos de parece de siera le venir allí Lotarioso ocasión Arba, sino que tiene
tiéne personas, y la casa! A lo dónígor una justición cerca en tantigo, saliendo fue la
manta que me alababa y él joya lo que fueran honida y vuestra caballería
de la gelosa,
q
(True, True)


## Ejercicios

1. Escribe la tabla de pesos de la media para una secuencia de cuatro posiciones. Propón una fila de puntuaciones, con -\infty en el futuro, que produzca la fila de la posición 3.

2. Calcula el producto escalar de [2,1] con [1,3] y con [-1,2]. ¿Cuál de las dos claves encaja mejor con la consulta [2,1]?

3. Añade al ejemplo una cuarta posición con consulta [1,1], clave [1,-1] y valor [3,1]. Sigue los cuatro pasos de la atención y calcula sus pesos de atención y su salida.

4. Cambia la consulta de la posición 3 del ejemplo por [0,1] y calcula sus nuevos pesos. ¿A qué posiciones atiende ahora sobre todo?

5. Si las consultas y las claves tuvieran 4 componentes, habría que dividir entre 4=2. Calcula softmax de [6, 0] y de [6/2, 0]. ¿Qué efecto tiene dividir entre la raíz?

6. Explica qué ocurriría durante el entrenamiento y al generar texto si quitáramos la máscara causal.

7. ¿Por qué hace falta el embedding de posición? Pon un ejemplo de dos textos que, sin él, la atención no podría distinguir.

8. Normaliza a mano el vector [1, 3, 5, 7]: calcula su media, su desviación típica y el resultado.

9. Explica con tus palabras qué ventaja tiene sumar la salida de cada bloque, x + f(x), en lugar de sustituir el vector por f(x).

10. Cuenta los parámetros de una cabeza, de un bloque y del modelo completo si los vectores tuvieran 128 números en lugar de 64, con las mismas 4 cabezas y una MLP que siga siendo cuatro veces más ancha (512).

11. Ejecuta el programa y anota la pérdida final de entrenamiento y de validación. Calcula e^-pérdida para tu modelo y para el bigrama, e interpreta la diferencia.

12. Comprobación de la máscara. Llama a `comprobar_mascara` cambiando el carácter de la posición 10 y después el de la posición 60. Explica los resultados y qué demuestran. No hace falta entrenar: la máscara funciona igual con un modelo recién creado, `GPT(92)`, así que puedes hacerlo en el cuaderno en unos segundos.

13. Cambia `CONTEXTO` a 16 y vuelve a entrenar. Compara la pérdida de validación con la del modelo de 64 caracteres y explica la diferencia.

## Referencias

Basado en la lección [Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY) y el código [ng-video-lecture](https://github.com/karpathy/ng-video-lecture) de Andrej Karpathy. Texto de *Don Quijote de la Mancha*, de dominio público. Explicaciones y ejercicios propios.